In [20]:
import torch
import pandas as pd
from tqdm import tqdm
import random
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, roc_curve, roc_auc_score
import matplotlib.pyplot as plt

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cpu


## Define attacks

In [3]:
from transformers import pipeline

# Back translation (en -> de -> en)
translator_en_de = pipeline("translation", model="Helsinki-NLP/opus-mt-en-de")
translator_de_en = pipeline("translation", model="Helsinki-NLP/opus-mt-de-en")

# Paraphraser (T5)
paraphraser = pipeline("text2text-generation", model="Vamsi/T5_Paraphrase_Paws")

# for style transfer
style_model = pipeline("text2text-generation", model="google/flan-t5-large")

c:\Users\reese\Documents\SUTD stuff\work stuff\term 6\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\reese\Documents\SUTD stuff\work stuff\term 6\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as ex

In [4]:
def back_translate(text):
    try:
        de = translator_en_de(text)[0]['translation_text']
        en = translator_de_en(de)[0]['translation_text']
        return en
    except:
        return text

def paraphrase(text, num_return_sequences=2):
    try:
        outputs = paraphraser(
            text,
            max_length=128,
            num_return_sequences=num_return_sequences,
            do_sample=True,
            top_k=50,
            top_p=0.95
        )
        return [o['generated_text'] for o in outputs]
    except:
        return [text] * num_return_sequences

def add_noise(text, prob=0.1):
    chars = list(text)
    for i in range(1, len(chars)):
        if random.random() < prob:
            chars[i], chars[i-1] = chars[i-1], chars[i]
    return "".join(chars)

# style transfer
def rewrite_to_human(text):
    prompt = f"While keeping the meaning the same, rewrite this to sound casual and human: {text}"
    return style_model(prompt)[0]['generated_text']

def rewrite_to_ai(text):
    prompt = f"While keeping the meaning the same, rewrite this to sound like it was written by AI: {text}"
    return style_model(prompt)[0]['generated_text']

In [11]:
def augment(text, label):
    choice = random.choices(["none", "style", "para", "bt", "noise"],
                            weights=[0.45, 0.25, 0.2, 0.05, 0.05])[0]

    if choice == "bt":
        return back_translate(text)
    elif choice == "para":
        return paraphrase(text, 1)[0]
    elif choice == "noise":
        return add_noise(text)
    elif choice == "style":
        if label == 0: # AI
            return rewrite_to_human(text)
        elif label == 1: # human
            return rewrite_to_ai(text)
    return text

## Dataset and Dataloader

In [8]:
# get original data
df = pd.read_csv("../cleaned_data/final_merged.csv")
rawtext = df["full_post_clean"].values
labels = df["Label"].values

In [6]:
# tokenizer
from transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [9]:
# Dataset
from torch.utils.data import Dataset

class PostsDataset_withAugData(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]

        text = augment(text, label)

        encoded_dict = tokenizer(
            text,                      
            add_special_tokens = True, 
            max_length = 256,          
            padding = 'max_length',    # Updated to the modern parameter name
            truncation = True,
            return_attention_mask = True,   
            return_tensors = 'pt'     
        )

        return {
            'input_ids': encoded_dict['input_ids'],
            'attention_masks': encoded_dict['attention_mask'],
            'labels': label
        }
    
    def __len__(self):
        return len(self.labels)
    
# Instantiate the full dataset
full_dataset = PostsDataset_withAugData(rawtext, labels)

In [18]:
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Subset

# Step A: Split into 70% Train and 30% Temporary (Validation + Test)
# We use stratify=labels.numpy() to ensure the 0/1 ratio remains perfectly balanced in all splits
train_idx, temp_idx = train_test_split(
    range(len(full_dataset)), 
    test_size=0.30, 
    stratify=labels, 
    random_state=42
)

# Step B: Split the 30% Temporary subset evenly in half to get 15% Validation and 15% Test
val_idx, test_idx = train_test_split(
    temp_idx, 
    test_size=0.50, 
    stratify=[labels[i].item() for i in temp_idx], 
    random_state=42
)

# 4. Create PyTorch Subsets
train_dataset = Subset(full_dataset, train_idx)
val_dataset = Subset(full_dataset, val_idx)
test_dataset = Subset(full_dataset, test_idx)

print(f"Total dataset size: {len(full_dataset)}")
print(f"Train size: {len(train_dataset)} (70%)")
print(f"Val size:   {len(val_dataset)} (15%)")
print(f"Test size:  {len(test_dataset)} (15%)")

# 5. Create DataLoaders
# Batch size of 16 is standard for BERT, reduce to 8 if you run out of GPU memory
batch_size = 32

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

Total dataset size: 51843
Train size: 36290 (70%)
Val size:   7776 (15%)
Test size:  7777 (15%)


## Model

In [ ]:
# model that we previously trained
from transformers import AutoConfig, AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
model.load_state_dict(torch.load("best_model.pt", weights_only=True))
model.to(device)

c:\Users\reese\Documents\SUTD stuff\work stuff\term 6\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

## Training functions (from model.ipynb)

In [21]:
# training function

def train(model, dataloader, optimizer, device):
    model.train()
    total_loss = 0
    total_preds = []
    total_labels = []

    for step, batch in enumerate(tqdm(dataloader, desc="Training")):
        # 1. MOVE TO GPU: Send the current batch of 32 rows to the GPU
        input_ids = batch["input_ids"].to(device)
        attention_masks = batch["attention_masks"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad()

        # 2. RUN ON GPU: The model processes the batch
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_masks,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits
        total_loss += loss.item() # .item() safely extracts the number from the GPU

        # 3. RUN ON GPU: Calculate gradients and update weights
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        # 4. GET PREDICTIONS ON GPU: Find the highest probability class (0 or 1)
        preds = torch.argmax(logits, dim=1)

        # 5. MOVE BACK TO CPU: 
        # .detach()  -> Unhooks the data from the GPU's gradient graph
        # .cpu()     -> Moves it from GPU VRAM back to standard computer RAM
        # .numpy()   -> Converts it from a PyTorch tensor to a standard Python/NumPy list
        total_preds.extend(preds.detach().cpu().numpy())
        total_labels.extend(labels.detach().cpu().numpy())

    # 6. RUN ON CPU: Scikit-learn calculates the final score using the CPU arrays
    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(total_labels, total_preds)
    f1score = f1_score(total_labels, total_preds)

    return avg_loss, accuracy, f1score

In [22]:
# evaluation function

def evaluate(model, dataloader, device):
    model.eval()
    total_loss = 0
    total_preds = []
    total_labels = []

    with torch.no_grad(): # Disables gradient tracking entirely to save GPU memory
        for batch in tqdm(dataloader, desc="Evaluating"):
            # 1. MOVE TO GPU
            input_ids = batch["input_ids"].to(device)
            attention_masks = batch["attention_masks"].to(device)
            labels = batch["labels"].to(device)

            # 2. RUN ON GPU
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_masks,
                labels=labels
            )

            loss = outputs.loss
            logits = outputs.logits
            total_loss += loss.item()

            # 3. GET PREDICTIONS ON GPU
            preds = torch.argmax(logits, dim=1)

            # 4. MOVE BACK TO CPU for safe storage and metric calculation
            total_preds.extend(preds.detach().cpu().numpy())
            total_labels.extend(labels.detach().cpu().numpy())

    # 5. RUN ON CPU
    avg_loss = total_loss / len(dataloader)
    accuracy = accuracy_score(total_labels, total_preds)
    f1score = f1_score(total_labels, total_preds)

    return avg_loss, accuracy, f1score

In [23]:
import os

class EarlyStopping:
    def __init__(self, patience, mode='max', delta=0, save_dir="checkpoints", filename="best_model.pt"):
        self.patience = patience
        self.mode = mode 
        self.delta = delta
        
        # Create the directory if it doesn't exist
        self.save_dir = save_dir
        if not os.path.exists(self.save_dir):
            os.makedirs(self.save_dir)
            print(f"Created directory: {self.save_dir}")
            
        self.path = os.path.join(self.save_dir, filename)
        self.counter = 0
        self.best_score = None
        self.early_stop = False

    def __call__(self, val_score, model):
        if self.best_score is None:
            self.best_score = val_score
            self.save_checkpoint(model)
            return

        if self.mode == 'max':
            if val_score < self.best_score + self.delta:
                self.counter += 1
                print(f"EarlyStopping counter: {self.counter}/{self.patience}")
                if self.counter >= self.patience:
                    self.early_stop = True
            else:
                self.best_score = val_score
                self.save_checkpoint(model)
                self.counter = 0
                
        elif self.mode == 'min':
            if val_score > self.best_score - self.delta:
                self.counter += 1
                print(f"EarlyStopping counter: {self.counter}/{self.patience}")
                if self.counter >= self.patience:
                    self.early_stop = True
            else:
                self.best_score = val_score
                self.save_checkpoint(model)
                self.counter = 0

    def save_checkpoint(self, model):
        # Saves the model state_dict to the specified directory/file
        print(f"Validation improved. Saving best model to {self.path}...")
        torch.save(model.state_dict(), self.path)

In [24]:
def training_loop(model, train_loader, val_loader, optimizer, early_stopping, num_epochs, device):
    train_losses, val_losses = [], []
    train_accs, val_accs = [], []
    train_f1s, val_f1s = [], []

    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")

        train_loss, train_acc, train_f1 = train(
            model, train_loader, optimizer, device
        )

        val_loss, val_acc, val_f1 = evaluate(
            model, val_loader, device
        )

        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.4f}")

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        train_accs.append(train_acc)
        val_accs.append(val_acc)
        train_f1s.append(train_f1)
        val_f1s.append(val_f1)

        early_stopping(val_acc, model)

        if early_stopping.early_stop:
            print("Early stopping triggered. Halting training.")
            break
            
    # Reload the best weights from the directory before finishing
    print(f"\nTraining Complete. Loading best weights from '{early_stopping.path}'...")
    model.load_state_dict(torch.load(early_stopping.path, weights_only=True))
    
    return train_losses, val_losses, train_accs, val_accs, train_f1s, val_f1s

## Training (from model.ipynb)

In [ ]:
DATA_DIR = "/content/drive/MyDrive/Colab Notebooks/data/cds" 

In [ ]:
import os


CHECKPOINT_DIR = f"{DATA_DIR}/bert_checkpoints_withAdversarial"
if not os.path.exists(CHECKPOINT_DIR):
    os.makedirs(CHECKPOINT_DIR)
    print(f"Created directory: {CHECKPOINT_DIR}")

# ==========================================
# PHASE 1: Train Only the Classification Head
# ==========================================
# Freeze all BERT parameters
for param in model.bert.parameters():
    param.requires_grad = False

# Keep only the final classifier layer trainable
for param in model.classifier.parameters():
    param.requires_grad = True

# Monitor validation accuracy (higher is better)
early_stopping = EarlyStopping(patience=3, mode='max', save_dir=CHECKPOINT_DIR, filename="phase1_head_withAdversarial.pt")
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

# Run Phase 1 (1 Epoch)
train_losses_1, val_losses_1, train_accs_1, val_accs_1, train_f1s_1, val_f1s_1 = training_loop(
    model, train_loader, val_loader, optimizer, early_stopping, 1, device
)

# ==========================================
# PHASE 2: Unfreeze Last 2 BERT Layers
# ==========================================
for param in model.bert.encoder.layer[-2:].parameters():
    param.requires_grad = True

# New EarlyStopping instance for this phase to track improvements properly
early_stopping = EarlyStopping(patience=3, mode='max', save_dir=CHECKPOINT_DIR, filename="phase2_unfrozen_2_withAdversarial.pt")
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5) # Lower learning rate

# Run Phase 2 (3 Epochs)
train_losses_2, val_losses_2, train_accs_2, val_accs_2, train_f1s_2, val_f1s_2 = training_loop(
    model, train_loader, val_loader, optimizer, early_stopping, 3, device
)

# ==========================================
# PHASE 3: Unfreeze Last 4 BERT Layers
# ==========================================
for param in model.bert.encoder.layer[-4:-2].parameters():
    param.requires_grad = True
    
early_stopping = EarlyStopping(patience=3, mode='max', save_dir=CHECKPOINT_DIR, filename="phase3_unfrozen_4_withAdversarial.pt")
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-6) # Even lower learning rate

# Run Phase 3 (3 Epochs)
train_losses_3, val_losses_3, train_accs_3, val_accs_3, train_f1s_3, val_f1s_3 = training_loop(
    model, train_loader, val_loader, optimizer, early_stopping, 3, device
)

## Plot train and val (from model.ipynb)

In [ ]:
# plot graphs

total_train_losses = train_losses_1 + train_losses_2 + train_losses_3
total_train_accs = train_accs_1 + train_accs_2 + train_accs_3
total_train_f1s = train_f1s_1 + train_f1s_2 + train_f1s_3
total_val_losses = val_losses_1 + val_losses_2 + val_losses_3
total_val_accs = val_accs_1 + val_accs_2 + val_accs_3
total_val_f1s = val_f1s_1 + val_f1s_2 + val_f1s_3
iters = list(range(1, len(total_train_losses) + 1))

# 3. Create the figure with 3 subplots
fig, ax = plt.subplots(1, 3, figsize=(18, 5))

# --- Plot 1: Loss ---
ax[0].plot(iters, total_train_losses, marker='o', color='blue', label='Training Loss')
ax[0].plot(iters, total_val_losses, marker='s', color='orange', linestyle='--', label='Validation Loss')
ax[0].set_title('Training & Validation Loss')
ax[0].set_xlabel('Epochs')
ax[0].set_ylabel('Loss')
ax[0].grid(True)
ax[0].legend()

# --- Plot 2: Accuracy ---
ax[1].plot(iters, total_train_accs, marker='o', color='green', label='Training Accuracy')
ax[1].plot(iters, total_val_accs, marker='s', color='red', linestyle='--', label='Validation Accuracy')
ax[1].set_title('Training & Validation Accuracy')
ax[1].set_xlabel('Epochs')
ax[1].set_ylabel('Accuracy')
ax[1].grid(True)
ax[1].legend()

# --- Plot 3: F1 Score ---
ax[2].plot(iters, total_train_f1s, marker='o', color='purple', label='Training F1')
ax[2].plot(iters, total_val_f1s, marker='s', color='brown', linestyle='--', label='Validation F1')
ax[2].set_title('Training & Validation F1 Score')
ax[2].set_xlabel('Epochs')
ax[2].set_ylabel('F1 Score')
ax[2].grid(True)
ax[2].legend()

plt.tight_layout()


## Testing (from model.ipynb)

In [ ]:
def evaluate_on_testing_set(y_test, y_pred):
  # Calculate AUC
  print("AUC is: ", roc_auc_score(y_test, y_pred))

  # print out recall and precision
  print(classification_report(y_test, y_pred))

  # print out confusion matrix
  print("Confusion Matrix: \n", confusion_matrix(y_test, y_pred))

  # # calculate points for ROC curve
  fpr, tpr, thresholds = roc_curve(y_test, y_pred)

  # Plot ROC curve
  plt.plot(fpr, tpr, label='ROC curve (area = %0.3f)' % roc_auc_score(y_test, y_pred))
  plt.plot([0, 1], [0, 1], 'k--')  # random predictions curve
  plt.xlim([0.0, 1.0])
  plt.ylim([0.0, 1.0])
  plt.xlabel('False Positive Rate or (1 - Specifity)')
  plt.ylabel('True Positive Rate or (Sensitivity)')
  plt.title('Receiver Operating Characteristic')

In [ ]:
# testing

model.eval()

test_loss = 0
test_preds = []
test_labels = []

with torch.no_grad():

    for batch in tqdm(test_loader):

        input_ids = batch["input_ids"].to(device)
        attention_masks = batch["attention_masks"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_masks,
            labels=labels
        )

        loss = outputs.loss
        logits = outputs.logits

        test_loss += loss.item()

        preds = torch.argmax(logits, dim=1)

        test_preds.extend(preds.detach().cpu().numpy())
        test_labels.extend(labels.detach().cpu().numpy())

In [ ]:
# testing metrics

print("Test Loss:", test_loss)
evaluate_on_testing_set(test_labels, test_preds)